# HOGformer + soft-voting HOG (proposed) on Kaggle
Session options: **GPU T4 x2**, **Internet = On**, **Persistence = Files only**, Environment = latest.

**IMPORTANT - run it unattended with Save Version -> Save & Run All (Commit).** An interactive session whose notebook is idle
(training runs in a background process) is shut down by Kaggle after about an hour. If you stay interactive, you must keep a cell
busy by running the WAIT cell (5c) and keep the tab open.

To resume an interrupted run, attach a dataset with its `net_g_<N>.pth` and `<N>.state` (Add Input); cell 4 finds them.

In [ ]:
# 1) GPU and disk check
!nvidia-smi
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))
!df -h /tmp /kaggle/working

In [ ]:
# 2) Code + dependencies
%cd /kaggle/working
!test -d HOGformer_DLCV || git clone -b claude/lucid-fermi-emhnbo https://github.com/adityaj1706/HOGformer_DLCV.git
%cd /kaggle/working/HOGformer_DLCV
!git pull
!git log -1 --oneline
!pip install -q -r settingI/requirements_colab.txt
ALLWEATHER_ID = '1tfeBnjZX1wIhIFPl6HOzzOKOyo0GdGHl'   # replace with the id of YOUR shared copy if this hits the Drive quota

In [ ]:
# 3) Data -> /tmp/data (not /kaggle/working, which is capped and saved as output).
#    /content/data is a symlink so the configs are IDENTICAL to the Colab runs.
!mkdir -p /tmp/data /content
!ln -sfn /tmp/data /content/data
%cd /kaggle/working/HOGformer_DLCV
!SKIP_SNOW=1 ALLWEATHER_ID={ALLWEATHER_ID} bash scripts/download_data.sh /tmp/data

In [ ]:
# 4) RESUME (optional). If you attached a Kaggle dataset holding net_g_<N>.pth and <N>.state of an interrupted run,
#    this copies the newest iteration that has BOTH files. With no dataset attached it starts from scratch.
import glob, os, shutil, re
RUN_NAME = 'Allweather_HOGformer_softvote'
pths   = glob.glob('/kaggle/input/**/net_g_[0-9]*.pth', recursive=True)
states = glob.glob('/kaggle/input/**/*.state', recursive=True)
it = lambda p: int(re.findall(r'(\d+)', os.path.basename(p))[-1])
common = sorted({it(p) for p in pths} & {it(s) for s in states})
if not common:
    print('No checkpoint pair found -> training starts from iteration 0')
else:
    N = common[-1]; print('resuming from iteration', N)
    exp = f'/kaggle/working/experiments/{RUN_NAME}'
    os.makedirs(exp + '/models', exist_ok=True); os.makedirs(exp + '/training_states', exist_ok=True)
    shutil.copy([p for p in pths if it(p) == N][0], f'{exp}/models/net_g_{N}.pth')
    shutil.copy([s for s in states if it(s) == N][0], f'{exp}/training_states/{N}.state')
    print(os.listdir(exp + '/models'), os.listdir(exp + '/training_states'))

In [ ]:
# 5) START training in the BACKGROUND (this cell returns immediately).
#    Change CONFIG to ..._A.yml / ..._B.yml for the proposed models.
import subprocess, os
CONFIG = 'Allweather/Options/Allweather_HOGformer_softvote.yml'
os.chdir('/kaggle/working/HOGformer_DLCV/settingI')
!mkdir -p /kaggle/working/experiments   # must exist, or the link below dangles
!ln -sfn /kaggle/working/experiments experiments
proc = subprocess.Popen(f'python -u basicsr/train.py -opt {CONFIG} --launcher none > /kaggle/working/train_stdout.log 2>&1', shell=True)
print('started, pid', proc.pid)

In [ ]:
# 5b) MONITOR - run this cell any time (it does not interrupt training)
!tail -n 40 /kaggle/working/train_stdout.log | grep -E "iter: +[0-9]+0,|Validation|Resuming|End of training|Error|Traceback" | tail -4 | cut -c1-210
!nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv
print('still running' if proc.poll() is None else f'FINISHED, exit code {proc.returncode}')

In [ ]:
# 5c) WAIT until training finishes (needed for "Save & Run All"; optional when interactive)
import time
while proc.poll() is None:
    time.sleep(30)
print('training finished, exit code', proc.returncode)

In [ ]:
# 6) What was saved
!ls -la /kaggle/working/experiments/Allweather_HOGformer_softvote/models | tail -4
!grep -h Validation /kaggle/working/experiments/Allweather_HOGformer_softvote/train_*.log | tail -6